<a href="https://colab.research.google.com/github/RDGopal/IB9LQ-2026/blob/main/MLM7_Diffusion_Property_Risk_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conditional Diffusion for Property-Risk Image Generation

**Business scenario:** An insurer or lender wants to stress-test visual risk models using synthetic post-disaster property imagery.

This notebook trains a small conditional diffusion model on `post_images`. The damage class is the condition. Diffusion generates a **distribution of plausible risk scenes** under a specified business condition.

## Data setup

Upload `xbd_teaching_subset.zip` to the Colab runtime using the left-hand file browser, or place it in your current working directory.

Expected contents after extraction:

```text
xbd_teaching_subset/
  labels.csv
  pre_images/
  post_images/
```

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

ZIP_PATH = Path("xbd_teaching_subset.zip")
DATA_DIR = Path("xbd_teaching_subset")

if not ZIP_PATH.exists():
    matches = list(Path.cwd().rglob("xbd_teaching_subset.zip"))
    if matches:
        ZIP_PATH = matches[0]

if not DATA_DIR.exists():
    assert ZIP_PATH.exists(), (
        "Could not find xbd_teaching_subset.zip. "
        "Upload it to the current Colab runtime, then rerun this cell."
    )
    print("Extracting:", ZIP_PATH.resolve())
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(".")
else:
    print("Using existing extracted folder:", DATA_DIR.resolve())

if not (DATA_DIR / "labels.csv").exists() and Path("labels.csv").exists():
    DATA_DIR = Path(".")

labels = pd.read_csv(DATA_DIR / "labels.csv")
print("DATA_DIR:", DATA_DIR.resolve())
print("Rows:", len(labels))
display(labels.head())
display(labels["damage_label"].value_counts())

In [ ]:
import math, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(42)

DAMAGE_TO_ID = {"no-damage":0, "minor-damage":1, "major-damage":2, "destroyed":3}
ID_TO_DAMAGE = {v:k for k,v in DAMAGE_TO_ID.items()}

def load_image(path, size=64):
    img = Image.open(path).convert("RGB").resize((size, size))
    arr = np.array(img).astype("float32") / 255.0
    arr = arr * 2.0 - 1.0
    return torch.from_numpy(arr).permute(2, 0, 1)

def show_grid(xs, titles=None, ncols=4, figsize=(10,6)):
    xs = xs.detach().cpu().clamp(-1, 1)
    xs = (xs + 1) / 2
    n=len(xs); ncols=min(ncols,n); nrows=math.ceil(n/ncols)
    plt.figure(figsize=figsize)
    for i in range(n):
        plt.subplot(nrows,ncols,i+1); plt.imshow(xs[i].permute(1,2,0).numpy()); plt.axis('off')
        if titles is not None: plt.title(titles[i], fontsize=8)
    plt.tight_layout(); plt.show()

## Dataset

In [ ]:
class PostDamageDataset(Dataset):
    def __init__(self,data_dir,labels,split='train',image_size=64):
        self.data_dir=Path(data_dir); self.df=labels[labels['split'].eq(split)].reset_index(drop=True).copy(); self.image_size=image_size
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]; x=load_image(self.data_dir/row['post_image'],self.image_size); y=int(row['damage_id']); return x,y
IMAGE_SIZE=64; BATCH_SIZE=64
train_ds=PostDamageDataset(DATA_DIR,labels,'train',IMAGE_SIZE); val_ds=PostDamageDataset(DATA_DIR,labels,'val',IMAGE_SIZE)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,drop_last=True)
print('Train:',len(train_ds),'Val:',len(val_ds))
xb,yb=next(iter(train_loader)); show_grid(xb[:8],[ID_TO_DAMAGE[int(y)] for y in yb[:8]],ncols=4)

## Small conditional U-Net

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim): super().__init__(); self.dim=dim
    def forward(self,t):
        half=self.dim//2
        freqs=torch.exp(-math.log(10000)*torch.arange(half,device=t.device).float()/max(half-1,1))
        args=t.float()[:,None]*freqs[None,:]
        emb=torch.cat([torch.sin(args),torch.cos(args)],dim=-1)
        return F.pad(emb,(0,1)) if self.dim%2 else emb

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim):
        super().__init__()

        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)

        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)

        self.emb = nn.Linear(emb_dim, out_ch)

    def forward(self, x, emb):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        h = h + self.emb(emb)[:, :, None, None]

        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)
        return h
class TinyConditionalUNet(nn.Module):
    def __init__(self,num_classes=4,base=48,emb_dim=128):
        super().__init__()
        self.time_emb=nn.Sequential(SinusoidalTimeEmbedding(emb_dim),nn.Linear(emb_dim,emb_dim),nn.SiLU(),nn.Linear(emb_dim,emb_dim))
        self.class_emb=nn.Embedding(num_classes,emb_dim)
        self.down1=ConvBlock(3,base,emb_dim); self.down2=ConvBlock(base,base*2,emb_dim); self.down3=ConvBlock(base*2,base*4,emb_dim)
        self.pool=nn.AvgPool2d(2); self.up2=nn.ConvTranspose2d(base*4,base*2,2,2); self.up1=nn.ConvTranspose2d(base*2,base,2,2)
        self.dec2=ConvBlock(base*4,base*2,emb_dim); self.dec1=ConvBlock(base*2,base,emb_dim); self.out=nn.Conv2d(base,3,1)
    def forward(self,x,t,y):
        emb=self.time_emb(t)+self.class_emb(y)
        h1=self.down1(x,emb); h2=self.down2(self.pool(h1),emb); h3=self.down3(self.pool(h2),emb)
        u2=self.up2(h3); u2=self.dec2(torch.cat([u2,h2],1),emb)
        u1=self.up1(u2); u1=self.dec1(torch.cat([u1,h1],1),emb)
        return self.out(u1)

## DDPM utilities

In [ ]:
T=200
betas=torch.linspace(1e-4,0.02,T,device=DEVICE); alphas=1.0-betas; alphas_cumprod=torch.cumprod(alphas,dim=0)
sqrt_alphas_cumprod=torch.sqrt(alphas_cumprod); sqrt_one_minus_alphas_cumprod=torch.sqrt(1.0-alphas_cumprod)
def q_sample(x0,t,noise=None):
    if noise is None: noise=torch.randn_like(x0)
    a=sqrt_alphas_cumprod[t][:,None,None,None]; s=sqrt_one_minus_alphas_cumprod[t][:,None,None,None]
    return a*x0+s*noise
@torch.no_grad()
def sample_ddpm(model,y,n_steps=T,image_size=64):
    model.eval(); n=len(y); x=torch.randn(n,3,image_size,image_size,device=DEVICE); y=torch.tensor(y,dtype=torch.long,device=DEVICE)
    for i in reversed(range(n_steps)):
        t=torch.full((n,),i,device=DEVICE,dtype=torch.long); pred_noise=model(x,t,y)
        beta_t=betas[i]; alpha_t=alphas[i]; alpha_bar_t=alphas_cumprod[i]
        mean=(1/torch.sqrt(alpha_t))*(x-((1-alpha_t)/torch.sqrt(1-alpha_bar_t))*pred_noise)
        x=mean+torch.sqrt(beta_t)*torch.randn_like(x) if i>0 else mean
    return x.clamp(-1,1)

## Train

In [ ]:
model = TinyConditionalUNet(num_classes=4, base=64).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
EPOCHS=50
for epoch in range(1,EPOCHS+1):
    model.train(); losses=[]
    for x0,y in train_loader:
        x0=x0.to(DEVICE); y=y.to(DEVICE); t=torch.randint(0,T,(x0.size(0),),device=DEVICE).long(); noise=torch.randn_like(x0); xt=q_sample(x0,t,noise)
        pred=model(xt,t,y); loss=F.mse_loss(pred,noise)
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
    print(f'Epoch {epoch:02d} | loss={np.mean(losses):.4f}')

## Generate samples

In [ ]:
classes=[0,0,1,1,2,2,3,3]
samples=sample_ddpm(model,classes,n_steps=T,image_size=IMAGE_SIZE)
show_grid(samples,[ID_TO_DAMAGE[c] for c in classes],ncols=4)
Path('outputs').mkdir(exist_ok=True); torch.save(model.state_dict(),'outputs/mlm7_conditional_diffusion_xbd.pt')
print('Saved outputs/mlm7_conditional_diffusion_xbd.pt')

## Business interpretation

Diffusion is useful when the decision problem needs many plausible realizations of a damage condition, not one deterministic answer. Use cases include rare-event augmentation, claims stress testing, and classifier robustness.